In [4]:
import xarray as xr
import numpy as np
import h5py
from SlidingWindowDs import SlidingWindowDs

In [10]:
# --- Load your HDF5 into xarray ---
h5_path = "twoCA.h5"

with h5py.File(h5_path, "r") as f:
    print("Keys in HDF5 file:", list(f.keys()))
    dset = f["wca_exp_03"][:]  # loads into memory (OK for quick test)
    lat = np.arange(dset.shape[1])
    lon = np.arange(dset.shape[2])
    time = np.arange(dset.shape[0])

Keys in HDF5 file: ['wca_exp_03']


In [11]:
da = xr.DataArray(
    dset,
    dims=("time", "lat", "lon"),
    coords={"lat": lat, "lon": lon, "time": time}
)

In [12]:
# Make sure order is (time, lat, lon, feature)
da = da.transpose("time", "lat", "lon").expand_dims("feature")

In [13]:
# --- Create dataset ---
seq_length = 8
ds = SlidingWindowDs(da, seq_length=seq_length)
print(ds)

This Dataset has :
51 timestamps

        50 * 50 grid size
1 features



In [27]:
# --- Test first valid sample ---
print(len(ds), "samples in dataset")
nb_invalid = 0
for i in range(len(ds)):
    try:
        X, y = ds[i]
        #continue
        #print(f"Index {i} → X shape {X.shape}, Y shape {y.shape}")
        
    except IndexError:
        #print(f"Index {i} is invalid, trying next...")
        nb_invalid += 1
        continue
print(f"Total invalid indices encountered: {nb_invalid}")

105000 samples in dataset
Total invalid indices encountered: 8232
